### Batching de l'évaluation du contexte droit

Cette section détaille la mécanique de **vectorisation GPU** utilisée pour évaluer l'ensemble des mots du vocabulaire à la position masquée, en tenant compte du **contexte droit** (les mots qui suivent la cible).


#### 1. Pourquoi cette étape est-elle nécessaire ?

Dans un modèle de langage autoregressif à $N$-grammes (de taille de fenêtre $k$), la prédiction d'un mot dépend uniquement des $k$ mots qui le précèdent. 

Pour évaluer la pertinence d'un mot candidat $w$ à la position masquée, nous devons mesurer deux choses :
1. **Contexte gauche :** Quelle est la probabilité de générer $w$ sachant les $k$ mots précédents ? $P(w \mid \text{gauche})$
2. **Contexte droit :** À quel point le mot $w$ permet-il de prédire correctement les $k$ mots qui le suivent ? 

Pour évaluer le mot situé à un décalage $shift$ à droite (noté $w_{\text{droit}}$), le modèle a besoin de son propre historique de taille $k$. Cet historique contient le mot candidat $w$ à la position $k - shift$.



#### 2. Déroulement étape par étape (Exemple avec $k = 2$)

Considérons la séquence locale `[10, 101, Target, 20, 30]`. 
Pour $shift = 1$, nous vérifions si le candidat permet de bien prédire le mot suivant (`20`) :

1. **Extraction du patron (`context_template`) :** 
   On extrait la fenêtre de taille $k=2$ se terminant à la position masquée : `[101, Target]`.
2. **Ajout de la dimension de batch (`unsqueeze(0)`) :** 
   Le tenseur 1D `[101, Target]` de taille $(2,)$ devient un tenseur 2D `[[101, Target]]` de taille $(1, 2)$ pour respecter le format d'entrée attendu par PyTorch `(batch_size, sequence_length)`.
3. **Duplication du contexte (`repeat(batch_size, 1)`) :** 
   Pour évaluer $N$ candidats simultanément sans faire de boucle `for`, la ligne est répliquée $N$ fois. On obtient une matrice de taille $(N, 2)$ où chaque ligne est identique.
4. **Injection des candidats (`contexts[:, pos] = candidates`) :** 
   La colonne correspondant à la position masquée est remplacée par la liste des IDs des candidats.
5. **Évaluation vectorisée :** 
   La matrice résultante est envoyée au MLP en un*forward*, calculant ainsi $P(w_{\text{droit}} \mid \text{contexte})$ pour tous les candidats en parallèle.

In [3]:
import torch

# ==============================================================================
# DEMONSTRATION : Vectorisation de l'évaluation du contexte droit
# ==============================================================================

# --- CONTEXTE DE L'EXEMPLE ---
# Sequence complete (2k + 1 = 5 tokens) : [10, 101, Target, 20, 30]
# k = 2 (Taille de l'historique de mots en entree du MLP)
k = 2
local_tensor = torch.tensor([10, 101, -1, 20, 30])  # -1 = Target masquée

# On simule un décalage vers la droite : shift = 1
# Le mot cible à droite que l'on veut prédire est '20' (index k + shift = 3)
shift = 1
right_target_id = local_tensor[k + shift].item()  # ID = 20

# ------------------------------------------------------------------------------
# ÉTAPE 1 : Extraire le patron du contexte de taille k
# ------------------------------------------------------------------------------
# Pour prédire le mot '20' avec k=2, le MLP a besoin des 2 mots juste avant lui.
# Ces 2 mots sont : [101, Candidate]
context_template = local_tensor[shift : shift + k]

print("1. Patron initial (1D) :", context_template.tolist())
# Output: [101, -1]


# ------------------------------------------------------------------------------
# ÉTAPE 2 : Ajouter la dimension de Batch (Unsqueeze)
# ------------------------------------------------------------------------------
# PyTorch attend toujours une entrée sous forme (batch_size, sequence_length).
# On transforme la liste [101, -1] (taille k) en matrice 1x2 [[101, -1]].
x = context_template.unsqueeze(0)

print("2. Forme après unsqueeze(0) :", x.shape)
# Output: torch.Size([1, 2])


# ------------------------------------------------------------------------------
# ÉTAPE 3 : Dupliquer le contexte pour chaque candidat du Batch (.repeat)
# ------------------------------------------------------------------------------
# Au lieu de tester les candidats un par un dans une boucle, on va tester un
# lot de candidats en parallèle sur GPU.
# Supposons un lot de 4 candidats à tester :
current_batch_ids = torch.tensor([1000, 1001, 1002, 1003])
current_batch_len = len(current_batch_ids)

# .repeat(4, 1) signifie :
# - Axe 0 (lignes)   : Répéter 4 fois (1 fois par candidat)
# - Axe 1 (colonnes) : Répéter 1 seule fois (garder la taille de contexte k=2)
contexts = x.repeat(current_batch_len, 1)

print("\n3. Matrice créée par .repeat(4, 1) :")
print(contexts)
# Matrice résultante (4 x 2) :
# tensor([[101, -1],
#         [101, -1],
#         [101, -1],
#         [101, -1]])


# ------------------------------------------------------------------------------
# ÉTAPE 4 : Injecter les candidats dans la position masquée
# ------------------------------------------------------------------------------
# La position de la cible masquée dans cet historique glissant est :
candidate_position = k - shift  # Pour k=2, shift=1 -> position 1 (la 2e colonne)

# On remplace la 2e colonne (position 1) par la liste de nos 4 candidats
contexts[:, candidate_position] = current_batch_ids

print("\n4. Matrice finale prête à passer dans le MLP (contexts) :")
print(contexts)
# Matrice envoyée au modèle :
# tensor([[101, 1000],   <- Teste si [101, 1000] prédit bien '20'
#         [101, 1001],   <- Teste si [101, 1001] prédit bien '20'
#         [101, 1002],   <- Teste si [101, 1002] prédit bien '20'
#         [101, 1003]])  <- Teste si [101, 1003] prédit bien '20'


# ------------------------------------------------------------------------------
# ÉTAPE 5 : Évaluation vectorisée par le modèle
# ------------------------------------------------------------------------------
# En une seule passe avant (forward), le MLP calcule les log-probabilités
# pour les 4 hypothèses simultanément sur GPU :
#
# logits = model(contexts)                       # Forme: (4, vocab_size)
# log_probs = F.log_softmax(logits, dim=-1)      # Forme: (4, vocab_size)
# scores_candidats += log_probs[:, right_target_id] # Extrait P(20 | contexte)

1. Patron initial (1D) : [101, -1]
2. Forme après unsqueeze(0) : torch.Size([1, 2])

3. Matrice créée par .repeat(4, 1) :
tensor([[101,  -1],
        [101,  -1],
        [101,  -1],
        [101,  -1]])

4. Matrice finale prête à passer dans le MLP (contexts) :
tensor([[ 101, 1000],
        [ 101, 1001],
        [ 101, 1002],
        [ 101, 1003]])
